In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取存在数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.ads_hot_communities_daily 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_community_scatter_data(city_name):
    """
    获取指定城市的小区散点图数据。
    提取：小区名、平均总价(X)、总关注数(Y)、平均单价(Size/Color)。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        community, 
        AVG(avg_total_price) as total_price,
        MAX(total_attention) as attention,
        AVG(avg_unit_price) as unit_price
    FROM iceberg_catalog.ershoufang.ads_hot_communities_daily
    WHERE city = '{city_name}' 
      AND community IS NOT NULL 
      AND community != ''
      AND community != '未知'
    GROUP BY community
    HAVING total_price > 0 AND attention > 0 AND unit_price > 0
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_bubble_scatter_with_marginals(city):
    """绘制带边缘分布的气泡散点图 (利用 GridSpec)"""
    df = fetch_community_scatter_data(city)
    
    if df.empty or len(df) < 10:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 有效小区数据不足，无法生成散点图", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 数据清洗与防畸变处理 ---
    q_price = df['total_price'].quantile(0.99)
    q_attn = df['attention'].quantile(0.99)
    plot_df = df[(df['total_price'] <= q_price) & (df['attention'] <= q_attn)].copy()

    x = plot_df['total_price']
    y = plot_df['attention']
    z = plot_df['unit_price']  

    min_z, max_z = z.min(), z.max()
    sizes = 30 + ((z - min_z) / (max_z - min_z)) * 770

    # --- 2. 使用 GridSpec 搭建画布布局 ---
    fig = plt.figure(figsize=(12, 10))
    # 划分 4x4 网格，主图占 3x3，边缘图各占 1 行/列
    gs = gridspec.GridSpec(4, 4, hspace=0.1, wspace=0.1)

    # 分配坐标轴轴域
    ax_main = fig.add_subplot(gs[1:, :-1])         # 左下：主散点图
    ax_top = fig.add_subplot(gs[0, :-1], sharex=ax_main)  # 左上：X轴边缘分布
    ax_right = fig.add_subplot(gs[1:, -1], sharey=ax_main) # 右下：Y轴边缘分布

    # --- 3. 绘制主散点图 (气泡图) ---
    scatter = ax_main.scatter(
        x, y, 
        s=sizes,          # 气泡大小 -> 单价
        c=z,              # 气泡颜色 -> 单价
        cmap='viridis',   # 翠绿-深蓝渐变色
        alpha=0.65,      
        edgecolors='white', 
        linewidth=0.5
    )

    ax_main.set_xlabel('小区平均总价 (万元)', fontsize=12, labelpad=10)
    ax_main.set_ylabel('累计关注人数', fontsize=12, labelpad=10)
    ax_main.grid(color='#E5E7E9', linestyle='--', alpha=0.6)

    # 添加颜色条说明单价
    cbar = fig.colorbar(scatter, ax=ax_main, orientation='horizontal', fraction=0.05, pad=0.12)
    cbar.set_label('小区平均单价 (元/㎡)  [气泡颜色与大小均代表单价]', fontsize=11)

    # --- 4. 绘制边缘分布图 (直方图+KDE拟合) ---
    # 顶部：X轴(总价)的分布
    sns.histplot(x=x, ax=ax_top, color='#3498DB', kde=True, line_kws={'linewidth': 2})
    ax_top.set_ylabel('频数', fontsize=10)
    
    # 右侧：Y轴(关注度)的分布
    sns.histplot(y=y, ax=ax_right, color='#E74C3C', kde=True, line_kws={'linewidth': 2})
    ax_right.set_xlabel('频数', fontsize=10)

    # --- 5. 图表修饰与坐标轴清理 ---
    plt.setp(ax_top.get_xticklabels(), visible=False)
    plt.setp(ax_right.get_yticklabels(), visible=False)
    
    # 移除边缘图的多余边框
    sns.despine(ax=ax_top, top=True, right=True, left=False, bottom=True)
    sns.despine(ax=ax_right, top=True, right=True, left=True, bottom=False)
    ax_top.tick_params(axis='x', length=0)
    ax_right.tick_params(axis='y', length=0)

    # 整体标题定位到最上方
    fig.suptitle(f'[{city}] 小区“总价 vs 关注度”微观生态图', fontsize=18, fontweight='bold', y=0.96)

    plt.show()

In [ ]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                
                clear_output(wait=True)
                plot_bubble_scatter_with_marginals(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_bubble_scatter_with_marginals(default_city)

In [4]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=1, layout=Layout(width='250px'), options=('上海', '北京', '南京', '天津', '太原', …

Output()